# Netflix EDA — анализ каталога и пользовательских оценок

**Учебный групповой проект НИУ ВШЭ · команда №50 · 2025**  
Участники: Валерия Анастас, Анна Варнавская, Даниил Дудин, Даниэль Хусаинов.

**Бизнес-задача:** исследовать представленный каталог шоу и сформулировать гипотезы о связи характеристик контента, года выпуска и пользовательских оценок. Эта портфолио-версия систематизирует воспроизводимую часть командной работы и не приписывает весь проект одному участнику.

**Важно:** это EDA по выборке шоу, а не измерение всей библиотеки Netflix, реальных просмотров или успеха оригинальных проектов.

## 1. Источник данных

В репозитории опубликован `netflix_shows.csv`, экспортированный из приложенного преподавателем `NetflixShows.xlsx`: **1 000 исходных строк и 7 колонок**. В рамках исходной командной работы использовались также `netflix_titles.csv`, `df_combined.csv` и `all_netflix.csv`; эти файлы не приложены, поэтому здесь не воспроизводятся.

`rating` — возрастная классификация шоу (например, TV-MA); `user rating score` — числовая пользовательская оценка. Эти переменные нельзя смешивать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from IPython.display import display

sns.set_theme(style='whitegrid')
path = Path('netflix_shows.csv')
if not path.exists():
    raise FileNotFoundError('Поместите netflix_shows.csv рядом с Notebook')
raw = pd.read_csv(path)
required = {'title','rating','ratingLevel','ratingDescription','release year','user rating score','user rating size'}
assert required.issubset(raw.columns)
print('Исходных записей:', len(raw))
display(raw.head())

## 2. Качество данных и удаление дублей

Сравниваем точные повторы всех полей, долю пропусков и диапазоны числовых значений. Удаляем только полные дубли. Причина их возникновения **не доказана**: гипотеза о рекомендательном алгоритме, обсуждаемая в презентации, требует отдельной проверки.

In [ ]:
audit = pd.DataFrame({'type': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'missing_%': (raw.isna().mean()*100).round(1)})
display(audit)
print('Точных дублей:', raw.duplicated().sum())
print('Уникальных названий:', raw.title.nunique())
clean = raw.drop_duplicates().copy()
clean['release year'] = pd.to_numeric(clean['release year'],errors='coerce')
clean['user rating score'] = pd.to_numeric(clean['user rating score'],errors='coerce')
print('После дедупликации:', len(clean))
print('Нет числовой оценки:', clean['user rating score'].isna().sum())

## 3. Возрастные категории контента

Показываем частоту категорий `rating` в очищенной **выборке**, а не в полном каталоге Netflix.

In [ ]:
counts = clean.rating.fillna('Не указано').value_counts().head(12)
fig,ax = plt.subplots(figsize=(10,5))
sns.barplot(x=counts.values,y=counts.index,color='#c6284f',ax=ax)
ax.set(title='12 наиболее частых возрастных категорий',xlabel='Число записей',ylabel='Категория')
plt.tight_layout()
plt.show()
display((clean.rating.value_counts(normalize=True)*100).round(1).head(10).rename('Доля, %').to_frame())

## 4. Пользовательские оценки

Описываем только шоу, для которых `user rating score` известен. **Пропуск не равен нулю.** Распределение оценённой части выборки не доказывает удовлетворённости всех зрителей Netflix.

In [ ]:
scored = clean.dropna(subset=['user rating score']).copy()
print('С известной оценкой:',len(scored),'из',len(clean))
display(scored['user rating score'].describe().round(2).to_frame('Оценка'))
fig,ax=plt.subplots(figsize=(10,4))
sns.histplot(scored['user rating score'],bins=20,color='#244977',edgecolor='white',ax=ax)
ax.axvline(scored['user rating score'].median(),linestyle='--',color='#c6284f',label='Медиана')
ax.set(title='Распределение числовых оценок',xlabel='User rating score',ylabel='Шоу')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Годы выпуска

Год создания контента не равен году его добавления в Netflix и **не позволяет** говорить о производстве Netflix Originals.

In [ ]:
year_counts = clean['release year'].dropna().astype(int).value_counts().sort_index()
fig,ax=plt.subplots(figsize=(11,4))
ax.bar(year_counts.index,year_counts.values,color='#244977')
ax.set(title='Шоу по году выхода',xlabel='Год выхода',ylabel='Уникальных записей')
plt.tight_layout()
plt.show()
display(year_counts.sort_values(ascending=False).head(8).rename('Число').to_frame())

## 6. Сравнение оценок выпусков 2016 и 2017 годов

Отвечаем на один из вопросов исходного задания. Разность средних описывает **только выбранные записи**, а статистический тест не позволяет называть один год более успешным для бизнеса Netflix. Используем Welch t-test как исследовательскую проверку, без причинных утверждений.

In [ ]:
selected = scored[scored['release year'].isin([2016,2017])].copy()
display(selected.groupby('release year')['user rating score'].agg(['count','mean','median','std']).round(2))
s16 = selected.loc[selected['release year']==2016,'user rating score']
s17 = selected.loc[selected['release year']==2017,'user rating score']
test = ttest_ind(s17,s16,equal_var=False)
print(f'Разность средних (2017−2016): {s17.mean()-s16.mean():.2f}')
print(f'Welch p-value: {test.pvalue:.4f}')
fig,ax=plt.subplots(figsize=(7,4))
sns.boxplot(data=selected,x='release year',y='user rating score',order=[2016,2017],color='#99adc6',ax=ax)
ax.set(title='Оценки по годам',xlabel='Год',ylabel='Числовая оценка')
plt.tight_layout()
plt.show()

## 7. Описательный портрет шоу: Friends

В исходной командной презентации приведён пример сериала «Friends». Ниже используем **только поля, подтверждённые исходным CSV**; сведения о сезонах и зрителях из других источников не добавляем.

In [ ]:
example = clean.loc[clean.title.astype(str).str.fullmatch('Friends',case=False,na=False)]
if example.empty:
    print('Friends не найден в предоставленном файле')
else:
    display(example[['title','rating','release year','user rating score']].drop_duplicates())
    valid = example['user rating score'].dropna()
    if len(valid):
        x=valid.iloc[0]
        print(f'Числовая оценка: {x:g}; процент оценённых шоу выборки с оценкой не выше: {(scored["user rating score"]<=x).mean()*100:.1f}%')

## 8. Итоги и ограничения

- В исходном файле **1 000 записей**, **500 точных дублей**, после очистки — **500 записей**. Различий между точными дублями нет, но причина их появления неизвестна.
- Числовая оценка указана лишь у части очищенной выборки; **пропуски нельзя интерпретировать как низкие оценки**.
- `rating` — возрастная категория, `user rating score` — оценка пользователей.
- Годы выхода не являются доказательством изменения стратегии Netflix или количества произведённых компанией фильмов.
- Сравнение 2016/2017 — описательный анализ, который требует внешней проверки репрезентативности выборки.

**Расширение оригинальной командной работы:** анализ жанров, стран и форматов по `netflix_titles.csv`, а также LLM-предположения о зрителях по `df_combined.csv`. Эти данные не были приложены, и в портфолио они не выдаются за воспроизведённые факты. LLM-классификация **не является измерением реальной аудитории**.

Работа основана на командном проекте №50 НИУ ВШЭ; индивидуальные доли участия не документированы.